# Programmatic Flow Authoring Workshop

Learn to build docling-pipelines flows programmatically through hands-on exercises. This interactive workshop covers:

1. **Global Configuration** - Control flow-wide behavior
2. **Flow Validation** - Catch errors before execution
3. **Operator Patterns** - Common configurations that work
4. **Dynamic Generation** - Build flows based on parameters
5. **Advanced Patterns** - Branching and composition

## Prerequisites

- Ollama running on `http://localhost:11434`
- Model: `ollama pull nomic-embed-text`
- Virtual environment activated
- PYTHONPATH set to include `src/`

## What Makes This Unique

This notebook provides **hands-on learning** through interactive examples. For complete reference documentation, see:
- [Global Config Reference](../../docs/reference/GLOBAL_CONFIG.md)
- [Flow Authoring Format](../../docs/guides/FLOW_AUTHORING_FORMAT.md)
- [Python API Guide](../../docs/guides/PYTHON_API_GUIDE.md)

## Part 1: Setup and Prerequisites

In [ ]:
# Cell 1: Imports and Auto-Setup
import json
import os
import shutil
import sys
from pathlib import Path

from docpipe.lib.docpipe_flow_manager import DocpipeFlowManager

# Auto-configure PYTHONPATH if needed
if "PYTHONPATH" not in os.environ:
    src_path = Path.cwd().parent.parent / "src"
    sys.path.insert(0, str(src_path))
    print(f"Added to path: {src_path}")

print("✓ Imports loaded successfully")

In [ ]:
# Cell 2: Check Prerequisites
def check_prerequisites():
    """Check if required services are running"""
    import requests

    # Check Ollama
    try:
        response = requests.get("http://localhost:11434/api/tags", timeout=2)
        print("✓ Ollama is running")

        # Check for required model
        models = response.json().get("models", [])
        model_names = [m.get("name", "") for m in models]
        if any("nomic-embed-text" in name for name in model_names):
            print("✓ nomic-embed-text model available")
        else:
            print("⚠ nomic-embed-text model not found. Run: ollama pull nomic-embed-text")

        return True
    except Exception as e:
        print(f"✗ Ollama not running: {e}")
        print("  Start with: ollama serve")
        print("  Then run: ollama pull nomic-embed-text")
        return False


check_prerequisites()

In [ ]:
# Cell 3: Create Test Data
test_dir = Path("./temp_flow_authoring_test")
test_dir.mkdir(exist_ok=True)

# Create sample documents
(test_dir / "doc1.txt").write_text("This is the first sample document for testing flow authoring.")
(test_dir / "doc2.txt").write_text("This is the second sample document with different content.")
(test_dir / "doc3.txt").write_text("This is the third sample document for comprehensive testing.")

print(f"✓ Created test data in {test_dir}")
print(f"  Files: {list(test_dir.glob('*.txt'))}")

## Part 2: Global Configuration Fundamentals

Global configuration controls flow-wide behavior:
- **Execution**: How data is processed
- **Batching**: Parallel processing strategy
- **Storage**: Where data is stored
- **Incremental**: Track processed documents

Let's explore these interactively!

In [ ]:
# Cell 4: Minimal Flow with Extract Operator
print("=== Example 1: Minimal Flow (System Defaults) ===\n")

minimal_flow = {
    "flow_name": "minimal-flow",
    "global_config": {
        "disable_validation": True  # Skip validation for minimal example
    },
    "flow": [
        {
            "name": "ingest",
            "type": "ingest_source",
            "config": {"provider": "filesystem", "connection_params": {"paths": [str(test_dir)]}},
        },
        {
            "name": "extract",
            "type": "extract_operator",
            "depends_on": ["ingest"],
            "config": {"text_extraction": {"provider": "docling_library"}, "entity_extraction": {"provider": "none"}},
        },
    ],
}

print("Flow definition:")
print(json.dumps(minimal_flow, indent=2))

manager = DocpipeFlowManager(flow_def=minimal_flow)
print("\nExecuting with default global config...")
manager.execute()
print("✓ Executed successfully with system defaults")

In [ ]:
# Cell 5: Add Basic Global Config
print("=== Example 2: Basic Global Config ===")

basic_flow = {
    "flow_name": "basic-config-flow",
    "global_config": {
        "doc_column": "content",  # Which column contains document text
        "disable_validation": True,  # Skip validation (faster, riskier)
        "force_ingest": True,  # Reprocess all documents
    },
    "flow": [
        {
            "name": "ingest",
            "type": "ingest_source",
            "config": {"provider": "filesystem", "connection_params": {"paths": [str(test_dir)]}},
        },
        {
            "name": "extract",
            "type": "extract_operator",
            "depends_on": ["ingest"],
            "config": {"text_extraction": {"provider": "docling_library"}, "entity_extraction": {"provider": "none"}},
        },
    ],
}

print("\nGlobal config added:")
print(json.dumps(basic_flow["global_config"], indent=2))

manager = DocpipeFlowManager(flow_def=basic_flow)
manager.execute()
print("\n✓ Executed with custom global config")

In [ ]:
# Cell 6: Incremental Processing with File Modification Demo
print("=== Example 3: Incremental Processing ===")
print("\nIncremental mode: Only process new/changed documents\n")

import time
import uuid

incremental_flow = {
    "flow_name": "incremental-demo",
    "global_config": {
        "force_ingest": False,  # Enable incremental mode
        "retain_deleted_docs": True,  # Keep deleted docs in output
        "disable_validation": False,
    },
    "flow": [
        {
            "name": "ingest",
            "type": "ingest_source",
            "config": {"provider": "filesystem", "connection_params": {"paths": [str(test_dir)]}},
        },
        {
            "name": "extract",
            "type": "extract_operator",
            "depends_on": ["ingest"],
            "config": {"text_extraction": {"provider": "docling_library"}, "entity_extraction": {"provider": "none"}},
        },
    ],
}

# Generate a single job_id and reuse it for all runs
job_id = str(uuid.uuid4())
print(f"Using job_id: {job_id}")
print(f"Metadata location: ./data/{job_id}/incremental_metadata/\n")

print("Configuration:")
print(f"  force_ingest: {incremental_flow['global_config']['force_ingest']}")
print(f"  retain_deleted_docs: {incremental_flow['global_config']['retain_deleted_docs']}")

# First run: Process all documents
print("\n--- Run 1: Processing all documents ---")
manager = DocpipeFlowManager(flow_def=incremental_flow, job_id=job_id)
manager.execute()
print("✓ Run 1 complete: All 3 documents processed")

# Second run: Should skip unchanged documents
print("\n--- Run 2: Should skip all unchanged documents ---")
manager2 = DocpipeFlowManager(flow_def=incremental_flow, job_id=job_id)
manager2.execute()
print("✓ Run 2 complete: All documents skipped (no changes detected)")

# Modify one file to trigger incremental processing
print("\n--- Modifying doc1.txt to trigger incremental processing ---")
doc1_path = test_dir / "doc1.txt"
with doc1_path.open("a") as f:
    f.write("\n\nModified content added at " + time.strftime("%H:%M:%S"))
print(f"✓ Modified: {doc1_path}")

# Third run: Should only process the modified file
print("\n--- Run 3: Should only process modified document ---")
manager3 = DocpipeFlowManager(flow_def=incremental_flow, job_id=job_id)
manager3.execute()
print("✓ Run 3 complete: Only doc1.txt reprocessed (doc2.txt and doc3.txt skipped)")

print("\n" + "=" * 70)
print("Key Insight: Incremental processing detects file changes automatically")
print("=" * 70)
print("\nHow It Works:")
print("  1. Generate job_id once: job_id = str(uuid.uuid4())")
print("  2. Pass to all managers: DocpipeFlowManager(flow_def=..., job_id=job_id)")
print("  3. Metadata tracks file hashes and modification times")
print("  4. Only new/modified files are processed on subsequent runs")
print("  5. Metadata persists in ./data/<job_id>/incremental_metadata/")
print("\nPerformance Benefits:")
print("  - Run 1: Processes all 3 files")
print("  - Run 2: Skips all 3 files (6x faster!)")
print("  - Run 3: Processes only 1 modified file (3x faster!)")

In [ ]:
# Cell 7: Micro-Batching Demo
print("=== Example 4: Micro-Batching for Parallel Processing ===")
print("\nMicro-batching: Process documents in parallel batches\n")

batching_flow = {
    "flow_name": "batching-demo",
    "global_config": {
        "force_ingest": True,
        "enable_micro_batching": True,  # Enable parallel batching
        "micro_batch_size": 2,  # 2 documents per batch
        "disable_validation": True,
    },
    "flow": [
        {
            "name": "ingest",
            "type": "ingest_source",
            "config": {"provider": "filesystem", "connection_params": {"paths": [str(test_dir)]}},
        },
        {
            "name": "extract",
            "type": "extract_operator",
            "depends_on": ["ingest"],
            "config": {"text_extraction": {"provider": "docling_library"}, "entity_extraction": {"provider": "none"}},
        },
    ],
}

print("Batching config:")
print(f"  enable_micro_batching: {batching_flow['global_config']['enable_micro_batching']}")
print(f"  micro_batch_size: {batching_flow['global_config']['micro_batch_size']}")
print("\nWith 3 documents and batch_size=2:")
print("  Batch 1: doc1.txt, doc2.txt (parallel)")
print("  Batch 2: doc3.txt")

manager = DocpipeFlowManager(flow_def=batching_flow)
manager.execute()
print("\n✓ Processed documents in batches")
print("\nKey Insight: Batching improves throughput for large datasets")
print("  - Smaller batches = more parallelism, higher memory usage")
print("  - Larger batches = less parallelism, lower memory usage")

In [ ]:
# Cell 8: Storage Configuration
print("=== Example 5: Storage Configuration ===")

storage_flow = {
    "flow_name": "storage-demo",
    "global_config": {
        "data_storage_type": "local",  # Store on disk (vs memory)
        "data_local_config": {"output_folder": "./custom_output"},
        "disable_validation": False,
        "force_ingest": True,
    },
    "flow": [
        {
            "name": "ingest",
            "type": "ingest_source",
            "config": {"provider": "filesystem", "connection_params": {"paths": [str(test_dir)]}},
        },
        {
            "name": "extract",
            "type": "extract_operator",
            "depends_on": ["ingest"],
            "config": {"text_extraction": {"provider": "docling_library"}, "entity_extraction": {"provider": "none"}},
        },
    ],
}

print("\nStorage config:")
print(f"  Type: {storage_flow['global_config']['data_storage_type']}")
print(f"  Output: {storage_flow['global_config']['data_local_config']['output_folder']}")

manager = DocpipeFlowManager(flow_def=storage_flow)
manager.execute()
print("\n✓ Output saved to custom_output/")

# Verify storage
output_path = Path("./custom_output")
if output_path.exists():
    files = list(output_path.rglob("*.parquet"))
    print(f"  Found {len(files)} parquet file(s)")
    if files:
        print(f"  Example: {files[0].relative_to(output_path)}")
else:
    print("  ⚠ Output directory not found")
print("\nKey Insight: Use 'local' storage for large datasets, 'memory' for speed")

### Global Config Quick Reference

| Parameter | Values | Purpose |
|-----------|--------|----------|
| `force_ingest` | true/false | Reprocess all documents |
| `enable_micro_batching` | true/false | Enable parallel batch processing |
| `micro_batch_size` | integer | Documents per batch |
| `max_concurrent_batches` | integer | Parallel batch limit |
| `data_storage_type` | "memory"/"local" | Where to store data |
| `disable_validation` | true/false | Skip validation (risky) |
| `retain_deleted_docs` | true/false | Keep deleted docs |

**Full Reference**: [docs/reference/GLOBAL_CONFIG.md](../../docs/reference/GLOBAL_CONFIG.md)

## Part 3: Flow Validation

Always validate flows before execution to catch configuration errors early.

In [ ]:
# Cell 9: Valid Flow Example
print("=== Example 5: Valid Flow ===")

valid_flow = {
    "flow_name": "valid-flow",
    "flow": [
        {
            "name": "ingest",
            "type": "ingest_source",
            "config": {"provider": "filesystem", "connection_params": {"paths": [str(test_dir)]}},
        },
        {
            "name": "extract",
            "type": "extract_operator",
            "depends_on": ["ingest"],
            "config": {"text_extraction": {"provider": "docling_library"}, "entity_extraction": {"provider": "none"}},
        },
        {
            "name": "chunker",
            "type": "chunker",
            "depends_on": ["extract"],
            "config": {
                "provider": "docling",
                "doc_column": "content",
                "chunk_size": 512,
                "chunk_overlap": 50,
                "provider_config": {"api_base": "http://localhost:5000"},
            },
        },
        {
            "name": "embeddings",
            "type": "embeddings",
            "depends_on": ["chunker"],
            "config": {
                "provider": "litellm",
                "doc_column": "content",
                "embeddings_column": "embeddings",
                "provider_config": {
                    "model_id": "openai/nomic-embed-text",
                    "api_base": "http://localhost:11434/v1",
                    "api_key": "<your-api-key-here>",
                },
            },
        },
        {
            "name": "vectordb",
            "type": "vectordb",
            "depends_on": ["embeddings"],
            "config": {
                "provider": "opensearch",
                "index_name": "docpipe_demo",
                "doc_id_column": "doc_id_hash",
                "embeddings_column": "embeddings",
                "vector_dimension": 768,
                "create_index": True,
                "provider_config": {
                    "host": "localhost",
                    "port": 9200,
                    "username": "admin",
                    "password": "admin",
                    "use_ssl": False,
                    "verify_certs": False,
                    "engine": "faiss",
                    "algorithm": "hnsw",
                    "space_type": "l2",
                },
            },
        },
    ],
}

manager = DocpipeFlowManager(flow_def=valid_flow)
result = manager.validate()

print(f"\nValid: {result['valid']}")
print(f"Errors: {result['errors']}")
print(f"Warnings: {result['warnings']}")

if result["valid"]:
    print("\n✓ Flow is valid and ready to execute")
    print("\nNote: This complete RAG pipeline includes:")
    print("  1. Document ingestion (ingest_source)")
    print("  2. Text extraction (extract_operator)")
    print("  3. Document chunking (chunker)")
    print("  4. Embedding generation (embeddings via litellm + Ollama)")
    print("  5. Vector storage (vectordb with OpenSearch)")
    print("\nRequired services:")
    print("  - Docling API at http://localhost:5000")
    print("  - Ollama at http://localhost:11434 (with nomic-embed-text model)")
    print("  - OpenSearch at http://localhost:9200")
else:
    print("\n✗ Flow validation failed")

In [ ]:
# Cell 10: Common Validation Errors
print("=== Example 6: Common Validation Errors ===")
print("Note: These examples INTENTIONALLY show validation errors to demonstrate what to avoid\n")

# Error 1: Missing ingest operator
print("Error 1: Missing Ingest Operator (EXPECTED TO FAIL)")
error_flow_1 = {
    "flow_name": "missing-ingest",
    "flow": [
        {
            "name": "extract",
            "type": "extract_operator",
            "config": {"text_extraction": {"provider": "docling_library"}, "entity_extraction": {"provider": "none"}},
        }
    ],
}
result = DocpipeFlowManager(flow_def=error_flow_1).validate()
print(f"  Valid: {result['valid']} (Expected: False)")
if result["errors"]:
    error = result["errors"][0]
    # Handle both dict and string formats
    if isinstance(error, dict):
        error_msg = error.get("message", str(error))
    else:
        error_msg = error
    print(f"  Error: {error_msg}")
    print("  Lesson: Always start flows with an ingest operator")

# Error 2: Invalid dependency (catches compilation exception)
print("\nError 2: Invalid Dependency (EXPECTED TO FAIL)")
error_flow_2 = {
    "flow_name": "invalid-dependency",
    "flow": [
        {
            "name": "ingest",
            "type": "ingest_source",
            "config": {"provider": "filesystem", "connection_params": {"paths": [str(test_dir)]}},
        },
        {
            "name": "extract",
            "type": "extract_operator",
            "depends_on": ["nonexistent"],  # Invalid!
            "config": {"text_extraction": {"provider": "docling_library"}, "entity_extraction": {"provider": "none"}},
        },
    ],
}
try:
    result = DocpipeFlowManager(flow_def=error_flow_2).validate()
    print(f"  Valid: {result['valid']} (Expected: False)")
    if result["errors"]:
        error = result["errors"][0]
        if isinstance(error, dict):
            error_msg = error.get("message", str(error))
        else:
            error_msg = error
        print(f"  Error: {error_msg}")
except Exception as e:
    print("  Valid: False (Expected: False)")
    # Extract just the error message part
    error_text = str(e)
    if "Authoring flow validation failed:" in error_text:
        error_msg = error_text.split("Authoring flow validation failed:")[1].strip()
    else:
        error_msg = error_text
    print(f"  Error: {error_msg}")
print("  Lesson: Dependencies must reference existing operator names")

# Error 3: Missing extract operator
print("\nError 3: Missing Extract Operator (EXPECTED TO FAIL)")
error_flow_3 = {
    "flow_name": "missing-extract",
    "flow": [
        {
            "name": "ingest",
            "type": "ingest_source",
            "config": {"provider": "filesystem", "connection_params": {"paths": [str(test_dir)]}},
        },
        {"name": "language", "type": "lang_detect", "depends_on": ["ingest"], "config": {"provider": "fasttext"}},
    ],
}
result = DocpipeFlowManager(flow_def=error_flow_3).validate()
print(f"  Valid: {result['valid']} (Expected: False)")
if result["errors"]:
    error = result["errors"][0]
    # Handle both dict and string formats
    if isinstance(error, dict):
        error_msg = error.get("message", str(error))
    else:
        error_msg = error
    print(f"  Error: {error_msg}")
    print("  Lesson: Extract operator must follow ingest operator")

# Error 4: Duplicate operator names
print("\nError 4: Duplicate Operator Names (EXPECTED TO FAIL)")
error_flow_4 = {
    "flow_name": "duplicate-names",
    "flow": [
        {
            "name": "ingest",
            "type": "ingest_source",
            "config": {"provider": "filesystem", "connection_params": {"paths": [str(test_dir)]}},
        },
        {
            "name": "ingest",  # Duplicate!
            "type": "extract_operator",
            "depends_on": ["ingest"],
            "config": {"text_extraction": {"provider": "docling_library"}, "entity_extraction": {"provider": "none"}},
        },
    ],
}
try:
    result = DocpipeFlowManager(flow_def=error_flow_4).validate()
    print(f"  Valid: {result['valid']} (Expected: False)")
    if result["errors"]:
        error = result["errors"][0]
        if isinstance(error, dict):
            error_msg = error.get("message", str(error))
        else:
            error_msg = error
        print(f"  Error: {error_msg}")
except Exception as e:
    print("  Valid: False (Expected: False)")
    # Extract just the error message part
    error_text = str(e)
    if "Authoring flow validation failed:" in error_text:
        error_msg = error_text.split("Authoring flow validation failed:")[1].strip()
    else:
        error_msg = error_text
    print(f"  Error: {error_msg}")
print("  Lesson: Each operator must have a unique name")

print("\n" + "=" * 80)
print("Key Insight: Validation catches errors BEFORE execution, saving time and resources")
print("All errors above are INTENTIONAL demonstrations of what NOT to do")
print("=" * 80)

In [ ]:
# Cell 11: Invalid Flow Example
print("=== Example 7: Invalid Flow (Learning from Errors) ===")

invalid_flow = {
    "flow_name": "invalid-flow",
    "flow": [
        {
            "name": "extract",
            "type": "extract_operator",
            "depends_on": ["nonexistent"],  # Invalid dependency!
            "config": {"text_extraction": {"provider": "docling_library"}, "entity_extraction": {"provider": "none"}},
        }
    ],
}

from docpipe.exceptions.docpipe_exceptions import FlowInvalidDataException

try:
    manager = DocpipeFlowManager(flow_def=invalid_flow)
    result = manager.validate()
    print(f"\nValid: {result['valid']}")
    print("\nErrors found:")
    for error in result["errors"]:
        print(f"  - {error}")
except FlowInvalidDataException as e:
    print("\nValid: False")
    print("\nErrors found:")
    print(f"  - {e}")

print("\nKey Insight: Validation catches errors before execution")

## Part 5: Dynamic Flow Building

Build flows programmatically based on parameters - the most powerful feature!

In [ ]:
def build_pipeline(
    source_path: str,
    enable_quality: bool = False,
    enable_chunking: bool = False,
    enable_embeddings: bool = False,
    batch_size: int = 100,
    force_reprocess: bool = False,
    storage_type: str = "memory",
    validate: bool = False,
):
    """Build a pipeline dynamically based on requirements"""

    flow = {
        "flow_name": "dynamic-pipeline",
        "global_config": {
            "doc_column": "content",
            "force_ingest": force_reprocess,
            "micro_batch_size": batch_size,
            "data_storage_type": storage_type,
            "disable_validation": not validate,  # Invert for clarity
        },
        "flow": [],
    }

    # Always start with ingestion
    flow["flow"].append(
        {
            "name": "ingest",
            "type": "ingest_source",
            "config": {"provider": "filesystem", "connection_params": {"paths": [source_path]}},
        }
    )

    # Extract operator must follow ingest
    flow["flow"].append(
        {
            "name": "extract",
            "type": "extract_operator",
            "depends_on": ["ingest"],
            "config": {"text_extraction": {"provider": "docling_library"}, "entity_extraction": {"provider": "none"}},
        }
    )

    last_op = "extract"

    # Conditionally add quality checks
    if enable_quality:
        flow["flow"].append(
            {"name": "quality", "type": "lang_detect", "depends_on": [last_op], "config": {"provider": "fasttext"}}
        )
        last_op = "quality"

    # Conditionally add chunking
    if enable_chunking:
        flow["flow"].append(
            {
                "name": "chunk",
                "type": "chunker",
                "depends_on": [last_op],
                "config": {"chunk_type": "simple", "chunk_size": 512, "chunk_overlap": 50},
            }
        )
        last_op = "chunk"

    # Conditionally add embeddings
    if enable_embeddings:
        flow["flow"].append(
            {
                "name": "embeddings",
                "type": "embeddings",
                "depends_on": [last_op],
                "config": {
                    "provider": "litellm",
                    "doc_column": "content",
                    "embeddings_column": "embeddings",
                    "provider_config": {
                        "model_id": "openai/nomic-embed-text",
                        "api_base": "http://localhost:11434/v1",
                        "api_key": "<your-api-key-here>",
                    },
                },
            }
        )

    return flow


print("✓ Dynamic flow builder function defined")
print("\nParameters:")
print("  - source_path: Where to find documents")
print("  - enable_quality: Add language detection")
print("  - enable_chunking: Add document chunking")
print("  - enable_embeddings: Add embedding generation")
print("  - batch_size: Documents per batch")
print("  - force_reprocess: Reprocess all documents")
print("  - storage_type: 'memory' or 'local'")

In [ ]:
print("=== Configuration 1: Fast Processing ===")
print("Use Case: Quick iteration during development\n")

fast_config = build_pipeline(str(test_dir), enable_quality=True, batch_size=10, storage_type="memory")

print("Generated flow:")
print(json.dumps(fast_config, indent=2))

print("\nExecuting fast config...")
manager = DocpipeFlowManager(flow_def=fast_config)
manager.execute()
print("✓ Fast config executed")

In [ ]:
print("=== Configuration 2: Complete Pipeline ===")
print("Use Case: Full processing with all stages\n")

complete_config = build_pipeline(
    str(test_dir),
    enable_quality=True,
    enable_chunking=True,
    enable_embeddings=True,
    batch_size=50,
    storage_type="local",
)

print("Generated flow:")
print(json.dumps(complete_config, indent=2))

print("\nExecuting complete pipeline...")
manager = DocpipeFlowManager(flow_def=complete_config)
manager.execute()
print("✓ Complete pipeline executed")

## Part 6: Advanced Patterns

Branching workflows and composition helpers for complex pipelines.

In [ ]:
print("=== Advanced Pattern 1: Quality-Based Branching ===\n")
print("Use Case: Route documents based on readability scores\n")

branching_flow = {
    "flow_name": "quality-branching-pipeline",
    "global_config": {"disable_validation": False, "force_ingest": True},
    "flow": [
        {
            "name": "ingest",
            "type": "ingest_source",
            "config": {"provider": "filesystem", "connection_params": {"paths": [str(test_dir)]}},
        },
        {
            "name": "extract",
            "type": "extract_operator",
            "depends_on": ["ingest"],
            "config": {"text_extraction": {"provider": "docling_library"}, "entity_extraction": {"provider": "none"}},
        },
        {
            "name": "lang_detect",
            "type": "lang_detect",
            "depends_on": ["extract"],
            "config": {"provider": "fasttext", "doc_column": "content", "lang_column": "lang_name"},
        },
        {
            "name": "readability",
            "type": "readability",
            "depends_on": ["lang_detect"],
            "config": {"readability_score_list": ["flesch_reading_ease"]},
        },
        {
            "name": "quality_branching",
            "type": "branching",
            "depends_on": ["readability"],
            "config": {
                "branches": {
                    "high_quality": {
                        "link_name": "High Quality Documents",
                        "criteria_json": {
                            "criteria_list": [{"variable": "flesch_reading_ease", "operator": ">", "value": 60}],
                            "logical_operator": "AND",
                        },
                    },
                    "low_quality": {
                        "link_name": "Low Quality Documents",
                        "criteria_json": {
                            "criteria_list": [{"variable": "flesch_reading_ease", "operator": "<=", "value": 60}],
                            "logical_operator": "AND",
                        },
                    },
                }
            },
        },
        {
            "name": "ml_enrichment_high",
            "type": "ml_enrichment",
            "depends_on": ["quality_branching.high_quality"],
            "config": {"doc_column": "content", "lang_column": "lang_name"},
        },
        {
            "name": "filter_low",
            "type": "sql_filter",
            "depends_on": ["quality_branching.low_quality"],
            "config": {
                "criteria_json": {
                    "criteria_list": [{"variable": "flesch_reading_ease", "operator": ">", "value": 0}],
                    "logical_operator": "AND",
                }
            },
        },
    ],
}

print("Branching flow structure:")
print(json.dumps(branching_flow, indent=2))

print("\nExecuting branching flow...")
manager = DocpipeFlowManager(flow_def=branching_flow)
manager.execute()
print("✓ Branching flow executed")

print("\nKey Insights:")
print("  - Use 'quality_branching.high_quality' to reference specific branches")
print("  - Branching enables conditional processing based on document properties")
print("  - Each branch can have different downstream operators")

In [ ]:
print("=== Advanced Pattern 2: Composition Helpers ===")
print("Use Case: Reusable pipeline components\n")


def add_quality_stage(flow: dict, depends_on: str) -> str:
    """Add quality assessment operators to a flow"""
    flow["flow"].extend(
        [
            {"name": "language", "type": "lang_detect", "depends_on": [depends_on], "config": {"provider": "fasttext"}},
            {"name": "readability", "type": "readability", "depends_on": ["language"], "config": {}},
        ]
    )
    return "readability"  # Return last operator name


# Use the helper
composed_flow = {
    "flow_name": "composed-pipeline",
    "global_config": {"disable_validation": True, "force_ingest": True},
    "flow": [
        {
            "name": "ingest",
            "type": "ingest_source",
            "config": {"provider": "filesystem", "connection_params": {"paths": [str(test_dir)]}},
        },
        {
            "name": "extract",
            "type": "extract_operator",
            "depends_on": ["ingest"],
            "config": {"text_extraction": {"provider": "docling_library"}, "entity_extraction": {"provider": "none"}},
        },
    ],
}

# Add quality stage
last_op = add_quality_stage(composed_flow, "extract")
print(f"Added quality stage, last operator: {last_op}")

print("\nComposed flow:")
print(json.dumps(composed_flow, indent=2))

print("\nKey Insight: Composition helpers make complex pipelines maintainable")

## Part 7: Reference and Cleanup

In [ ]:
# Cell 19: Common Configuration Patterns
print("=== Common Global Config Patterns ===")

patterns = {
    "Development (Fast iteration)": {
        "disable_validation": True,
        "force_ingest": True,
        "micro_batch_size": 10,
        "data_storage_type": "memory",
    },
    "Production (Reliable, scalable)": {
        "disable_validation": False,
        "force_ingest": False,
        "micro_batch_size": 100,
        "max_concurrent_batches": 20,
        "data_storage_type": "local",
    },
    "Testing (Reproducible)": {"force_ingest": True, "micro_batch_size": 5, "data_storage_type": "memory"},
}

for name, config in patterns.items():
    print(f"\n{name}:")
    print(json.dumps(config, indent=2))

In [ ]:
# Cell 20: Cleanup Test Data
print("=== Cleanup ===")

# Remove test directory
if test_dir.exists():
    shutil.rmtree(test_dir)
    print(f"✓ Removed {test_dir}")

# Remove custom output directory
if Path("custom_output").exists():
    shutil.rmtree("custom_output")
    print("✓ Removed custom_output/")

print("\n✓ Cleanup complete")

## Workshop Complete!

### What You Learned

1. ✅ **Global Configuration** - Control flow-wide behavior (execution, batching, storage)
2. ✅ **Incremental Processing** - Process only new/changed documents
3. ✅ **Flow Validation** - Catch errors before execution
4. ✅ **Operator Patterns** - Common configurations that work
5. ✅ **Dynamic Generation** - Build flows programmatically based on parameters
6. ✅ **Advanced Patterns** - Branching and composition for complex pipelines

### Next Steps

1. **Explore Other Notebooks**
   - [01_quickstart.ipynb](01_quickstart.ipynb) - Complete pipeline in 10 minutes
   - [02_operator_showcase.ipynb](02_operator_showcase.ipynb) - All operator categories
   - [04_embeddings_vectordb.ipynb](04_embeddings_vectordb.ipynb) - Vector search
   - [06_rag_pipeline.ipynb](06_rag_pipeline.ipynb) - End-to-end RAG

2. **Read Full Documentation**
   - [Global Config Reference](../../docs/reference/GLOBAL_CONFIG.md) - All 20+ parameters
   - [Flow Authoring Format](../../docs/guides/FLOW_AUTHORING_FORMAT.md) - Complete syntax
   - [Python API Guide](../../docs/guides/PYTHON_API_GUIDE.md) - Advanced usage
   - [Operator Reference](../../docs/reference/OPERATORS.md) - All operators

3. **Try With Your Data**
   - Modify `build_pipeline()` for your use case
   - Experiment with different global configs
   - Build custom composition helpers
   - Check [sample_flows/](../../sample_flows/) for production examples

### Key Takeaways

- **Global config** controls flow-wide behavior - use it wisely
- **Incremental processing** saves time on large datasets
- **Always validate** before executing to catch errors early
- **Dynamic generation** is powerful for conditional pipelines
- **Composition helpers** make complex pipelines maintainable

Happy flow authoring! 🚀